In [0]:
base_path = "/Volumes/nyc/default/"
baseline_path = base_path + "yellow_baseline/"

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType


yellowSchema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
    #,
    #StructField("cbd_congestion_fee", DoubleType(), True)
])


yellow_taxi_raw_df = (
    spark.read
    .format("parquet")
    .schema(yellowSchema)
    .load(baseline_path)
)

In [0]:
%pip install h3 

In [0]:
%pip install geopandas

In [0]:
%pip install folium

In [0]:
import h3
import geopandas as gpd
import folium
from shapely.geometry import mapping

# 1. 加载并转换坐标系 (确保列名匹配你的 Index: zone, LocationID, borough)
gdf = gpd.read_file("/Volumes/nyc/default/nyczone/taxi_zones/taxi_zones.shp").to_crs("EPSG:4326")

# 2. 定义 H3 转换函数 (使用你之前调通的质心方案，最稳健)
def get_h3_cell(geom, res=8):
    centroid = geom.centroid
    try:
        # 兼容 H3 4.x 和 3.x
        return h3.latlng_to_cell(centroid.y, centroid.x, res) if hasattr(h3, 'latlng_to_cell') else h3.geo_to_h3(centroid.y, centroid.x, res)
    except:
        return None

# 3. 生成数据
gdf['centroid_h3_cells'] = gdf['geometry'].apply(get_h3_cell)
# 这一步定义变量 h3_df_exploded
h3_df_exploded = gdf[['LocationID', 'borough', 'zone', 'centroid_h3_cells']].copy()


In [0]:
# 1. 将 Pandas DataFrame 转换为 Spark DataFrame
# 注意：转换时 Spark 会根据 Pandas 的数据自动推断 Schema
from pyspark.sql.window import Window 
from pyspark.sql.functions import row_number, col 

windowSpec = Window.partitionBy("VendorID","tpep_pickup_datetime", "tpep_dropoff_datetime").orderBy(col("tolls_amount").desc())

# 2. 增加行号并过滤
yellow_taxi_df = (
    yellow_taxi_raw_df
    .withColumn("row_num", row_number().over(windowSpec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

spark_h3_df = spark.createDataFrame(h3_df_exploded)


# 2. 执行 Join (现在两边都是 Spark DataFrame 了)
yellow_taxi_enriched_trips = yellow_taxi_df.join(
    spark_h3_df.select("LocationID", "centroid_h3_cells"), 
    on=yellow_taxi_df.PULocationID == spark_h3_df.LocationID, 
    how="inner"
)

# 3. 检查结果
# 注意：请确认字段名是 lpep_... 还是 tpep_...（根据你之前定义的 Schema 应该是 lpep）
yellow_taxi_enriched_trips.select("tpep_pickup_datetime", "PULocationID", "centroid_h3_cells").show(10)

#### Spatio-Temporal Hotspots 时间分桶

In [0]:
from pyspark.sql import functions as F

# 1. 提取时间特征并关联 H3 (假设你已经完成了 join 映射表的操作)
hotspot_base_df = yellow_taxi_enriched_trips.select(
    "centroid_h3_cells", 
    "tpep_pickup_datetime", 
    "VendorID"
).withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
 .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) \
 .withColumn("day_name", F.date_format("tpep_pickup_datetime", "EEEE"))

# 2. 简单的质量检查
hotspot_base_df.select("tpep_pickup_datetime", "pickup_hour", "day_name").show(5)

#### Spatial-Temporal Aggregation

In [0]:

# 按 H3 格子、星期、小时进行聚合
hotspot_stats = hotspot_base_df.groupBy("centroid_h3_cells", "day_name", "pickup_hour") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy(F.desc("trip_count"))

# 显示最繁忙的 Top 10 “时空桶”
hotspot_stats.show(10)


#### Peak Detection

In [0]:
from pyspark.sql.window import Window

# 定义窗口：按地理格子分组，按订单量倒序排列
window_spec = Window.partitionBy("centroid_h3_cells").orderBy(F.desc("trip_count"))

# 找出每个格子单量最高的那一个小时
peak_analysis = hotspot_stats \
    .withColumn("rank", F.rank().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .select("centroid_h3_cells", "day_name", "pickup_hour", "trip_count")

# 看看纽约最繁忙的那些格子，它们的高峰通常发生在几点？
peak_analysis.orderBy(F.desc("trip_count")).show(10)


#### Seaborn Heatmap

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# 1. 准备数据：为了画图清晰，我们先选出 Top 50 的 H3 热点格子
top_h3_cells = hotspot_stats.groupBy("centroid_h3_cells") \
    .agg(F.sum("trip_count").alias("total_trips")) \
    .orderBy(F.desc("total_trips")) \
    .limit(50) \
    .select("centroid_h3_cells")

# 2. 过滤并转化为 Pandas DataFrame
plot_data_spark = hotspot_stats.join(top_h3_cells, on="centroid_h3_cells", how="inner") \
    .select("centroid_h3_cells", "day_name", "pickup_hour", "trip_count")

plot_data_pd = plot_data_spark.toPandas()

# 3. 创建透视表 (Pivoting)：将数据整理成 Heatmap 格式
# 行是 H3 格子，列是小时
heatmap_data = plot_data_pd.pivot_table(
    index='centroid_h3_cells', 
    columns='pickup_hour', 
    values='trip_count', 
    aggfunc='sum'
).fillna(0) # 填充空值为 0

# 4. 绘图
plt.figure(figsize=(16, 8))
sns.heatmap(heatmap_data, cmap='YlOrRd', robust=True, xticklabels=2) # xticklabels=2 每两小时显示一个刻度
plt.title('2016 NYC Taxi Spatio-Temporal Hotspots (Top 50 H3 Cells)')
plt.xlabel('Pickup Hour (0-23)')
plt.ylabel('H3 Cell ID')
plt.xticks(rotation=0)
plt.show()

#### Kepler.gl / Folium

In [0]:
%pip install h3

In [0]:
import folium
import json
import h3
from pyspark.sql import functions as F

# 1. 准备数据：选择一个特定的时间段（例如：周五晚上 18:00 - 晚高峰）
friday_night_hotspots = hotspot_stats \
    .filter((F.col("day_name") == "Friday") & (F.col("pickup_hour") == 18)) \
    .orderBy(F.desc("trip_count")) \
    .limit(100) # 取 Top 100

plot_geo_pd = friday_night_hotspots.toPandas()

# 2. 初始化地图：以曼哈顿为中心
m = folium.Map(location=[40.7588, -73.9851], zoom_start=12, tiles='CartoDB positron')

# 3. 准备 GeoJSON 格式的六边形几何体
polygons = []
for index, row in plot_geo_pd.iterrows():
    h3_cell = row['centroid_h3_cells']
    trip_count = row['trip_count']
    
    # --- 关键修改处 ---
    # h3 v4 API 使用 cell_to_boundary
    coords = h3.cell_to_boundary(h3_cell) 
    # ------------------
    
    # Folium 需要的是 [[lng, lat], [lng, lat], ...] 的格式
    geojson_coords = [[lng, lat] for lat, lng in coords]
    geojson_coords.append(geojson_coords[0]) # 闭合多边形
    
    # 构造 GeoJSON 特征
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [geojson_coords]
        },
        "properties": {
            "h3_cell": h3_cell,
            "trip_count": trip_count
        }
    }
    polygons.append(feature)

# 创建 GeoJSON 图层
geo_json_data = {"type": "FeatureCollection", "features": polygons}

# 4. 添加 Choropleth 层 (分级统计图)
folium.Choropleth(
    geo_data=geo_json_data,
    data=plot_geo_pd,
    columns=['centroid_h3_cells', 'trip_count'],
    key_on='feature.properties.h3_cell',
    fill_color='YlOrRd', # 黄-橙-红渐变
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Trip Count (Friday 18:00)'
).add_to(m)

# 显示地图
m

### 定义核心指标：每小时净收入 (Net Hourly Earnings)
在 2016 年，我们要衡量一个 H3 格子的“含金量”，不能只看总收入，要看时间产出比。

我们将计算：

Average Fare per Trip: 每一单平均能收多少钱。

Efficiency Index: 每分钟行程能赚多少钱（Fare / Trip Duration）。

In [0]:
from pyspark.sql import functions as F

# 1. 预处理：提取小时 + 计算时长 + 过滤异常值
efficiency_base = yellow_taxi_enriched_trips \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("duration_min", 
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    ) \
    .filter((F.col("duration_min") > 0.5) & (F.col("duration_min") < 180)) # 过滤掉0.5分钟以内的无效单和3小时以上的异常单

# 2. 按 H3 和 小时 聚合
revenue_stats = efficiency_base.groupBy("centroid_h3_cells", "pickup_hour") \
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        # 使用 expr 或 F.col 确保计算逻辑清晰
        F.avg(F.col("fare_amount") / F.col("duration_min")).alias("earnings_per_min")
    )

# 3. 查看结果
revenue_stats.orderBy(F.desc("earnings_per_min")).show(10)

### 寻找“黄金格子” (The Golden Hexagons)
我们要找出那些订单量不低，且每分钟收益最高的区域。

In [0]:
# 过滤掉订单量太少的格子（比如一小时少于 10 单的），避免统计偏差
golden_grids = revenue_stats.filter("trip_count > 10") \
    .orderBy(F.desc("earnings_per_min"))

print("2016年 赚钱效率最高的 Top 10 H3 格子：")
golden_grids.show(10)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 将 Spark DataFrame 聚合到小时维度（平均所有格子）并转为 Pandas
hourly_trend_pd = revenue_stats.groupBy("pickup_hour") \
    .agg(F.avg("earnings_per_min").alias("avg_efficiency")) \
    .orderBy("pickup_hour") \
    .toPandas()

# 2. 绘图
plt.figure(figsize=(10, 5))
plt.plot(hourly_trend_pd['pickup_hour'], hourly_trend_pd['avg_efficiency'], marker='o', linestyle='-', color='green')
plt.title('2016 Average Earnings per Minute by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('USD per Minute')
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24))
plt.show()

In [0]:
# 1. 取出你代码中过滤出的 golden_grids 前 15 名
top_grids_pd = golden_grids.limit(15).toPandas()

# 2. 绘图
plt.figure(figsize=(12, 6))
plt.bar(top_grids_pd['centroid_h3_cells'].astype(str), top_grids_pd['earnings_per_min'], color='orange')
plt.title('Top 15 Most Efficient H3 Cells (Golden Grids)')
plt.ylabel('Earnings per Minute ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [0]:
# 1. 取出你代码中过滤出的 golden_grids 前 15 名
top_grids_pd = golden_grids.limit(15).toPandas()

# 2. 绘图
plt.figure(figsize=(12, 6))
plt.bar(top_grids_pd['centroid_h3_cells'].astype(str), top_grids_pd['earnings_per_min'], color='orange')
plt.title('Top 15 Most Efficient H3 Cells (Golden Grids)')
plt.ylabel('Earnings per Minute ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [0]:
import folium
import json
import h3
from pyspark.sql import functions as F

# 1. 准备数据：选择一个特定的时间段（例如：周五晚上 18:00 - 晚高峰）
friday_night_hotspots = hotspot_stats \
    .filter((F.col("day_name") == "Friday") & (F.col("pickup_hour") == 18)) \
    .orderBy(F.desc("trip_count")) \
    .limit(100) # 取 Top 100

plot_geo_pd = friday_night_hotspots.toPandas()

# 2. 初始化地图：以曼哈顿为中心
m = folium.Map(location=[40.7588, -73.9851], zoom_start=12, tiles='CartoDB positron')

# 3. 准备 GeoJSON 格式的六边形几何体
polygons = []
for index, row in plot_geo_pd.iterrows():
    h3_cell = row['centroid_h3_cells']
    trip_count = row['trip_count']
    
    # --- 关键修改处 ---
    # h3 v4 API 使用 cell_to_boundary
    coords = h3.cell_to_boundary(h3_cell) 
    # ------------------
    
    # Folium 需要的是 [[lng, lat], [lng, lat], ...] 的格式
    geojson_coords = [[lng, lat] for lat, lng in coords]
    geojson_coords.append(geojson_coords[0]) # 闭合多边形
    
    # 构造 GeoJSON 特征
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [geojson_coords]
        },
        "properties": {
            "h3_cell": h3_cell,
            "trip_count": trip_count
        }
    }
    polygons.append(feature)

# 创建 GeoJSON 图层
geo_json_data = {"type": "FeatureCollection", "features": polygons}

# 4. 添加 Choropleth 层 (分级统计图)
folium.Choropleth(
    geo_data=geo_json_data,
    data=plot_geo_pd,
    columns=['centroid_h3_cells', 'earnings_per_min'],
    key_on='feature.properties.h3_cell',
    fill_color='RdYlGn', # 黄-橙-红渐变
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Trip Count (Friday 18:00)'
).add_to(m)

# 显示地图
m